# Expense Tracker: Data Cleaning Pipeline

This notebook demonstrates the cleaning and validation stage for the synthetic Expense Tracker dataset. It follows the current `src.data_cleaner.DataCleaner` API and does not fabricate executed outputs.

Run it from the project root:

```bash
jupyter notebook notebooks/02_data_cleaning.ipynb
```

If `data/raw_expenses.csv` does not exist, the first code cell creates it with the current generator so the notebook remains runnable on a fresh checkout.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import RAW_DATA_PATH, CLEANED_DATA_PATH, NUM_RECORDS, RANDOM_SEED
from src.data_generator import DataGenerator
from src.data_cleaner import DataCleaner

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw path: {RAW_DATA_PATH}")
print(f"Cleaned path: {CLEANED_DATA_PATH}")


In [ ]:
raw_path = Path(RAW_DATA_PATH)
if not raw_path.exists():
    generator = DataGenerator(num_records=NUM_RECORDS, seed=RANDOM_SEED)
    generator.generate_data()
    generator.save_data(RAW_DATA_PATH)
    print("Raw synthetic data was generated for this notebook.")
else:
    print("Using the existing raw synthetic data file.")

raw_data = pd.read_csv(RAW_DATA_PATH)
print(f"Raw shape: {raw_data.shape}")
raw_data.head()


In [ ]:
print("Missing values before cleaning:")
print(raw_data.isna().sum())
print(f"\nExact duplicate rows: {raw_data.duplicated().sum()}")
print("\nRaw data types:")
print(raw_data.dtypes)


In [ ]:
# DataCleaner reads the CSV in its constructor. Its clean_data method
# accepts remove_outliers=False; it flags outliers by default.
cleaner = DataCleaner(RAW_DATA_PATH)
cleaned_data = cleaner.clean_data(remove_outliers=False)
print(f"Cleaned shape: {cleaned_data.shape}")
cleaned_data.head()


In [ ]:
print("Cleaning report:")
print(cleaner.get_cleaning_report())

if "Is_Outlier" in cleaned_data.columns:
    print(f"Outliers flagged: {int(cleaned_data['Is_Outlier'].sum())}")

print("\nCleaned data types:")
print(cleaned_data.dtypes)


In [ ]:
output_path = cleaner.save_cleaned_data(CLEANED_DATA_PATH)
print(f"Saved cleaned data to: {output_path}")
print(f"File exists: {Path(output_path).exists()}")


In [ ]:
quality = {
    "rows": len(cleaned_data),
    "columns": len(cleaned_data.columns),
    "missing_cells": int(cleaned_data.isna().sum().sum()),
    "duplicate_rows": int(cleaned_data.duplicated().sum()),
    "non_positive_amounts": int((cleaned_data["Amount"] <= 0).sum()),
    "future_dates": int((cleaned_data["Date"] > pd.Timestamp.now()).sum()),
}
quality
